# Demo lesson 1: language patterns that change over time

A chat is text with a clock attached. Lesson 1 gives you two tools for that: a regular
expression turns a phrase into a column, and a `Pipeline` records which columns you made
and in what order. This demo uses both to answer three questions about one chat, and it
follows the six stages of the `goad` analysis checklist in order, because the order is the
method: write the question and the prediction down *before* looking, so that a plot can
still prove you wrong.

The chat is my own export, anonymised with `humanize`. Every number below is an aggregate:
no message text is shown, only the short phrase a regex matched.

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.datatransforms import (
    CountValues,
    Filter,
    GroupAgg,
    Pipeline,
    RegexFeature,
    SortValues,
    TimeFeatures,
    TransformBase,
)
from goad_toolkit.visualizer import (
    Annotate,
    BarPlot,
    FacetPlot,
    HorizontalLine,
    LinePlot,
    PlotSettings,
    ScatterPlot,
)

from wa_analyzer.data import load_own_chat
from wa_analyzer.humanhasher import humanize

## 1. Question: what do I want to know, and what would I expect?

Three questions, each with a prediction written down before any plot. A prediction you
write afterwards is a story; one you write first is a test.

| question | prediction, before looking |
|---|---|
| **Q1** How often is a variant of *ik mis je* said, and does that change over time? | The rate per message goes *up* over the months. Second guess: it is higher in weeks with *fewer* messages, because a quiet week is a week spent together and a busy one is a week apart, or the reverse. I do not know which, which is why it is a question. |
| **Q2** When is *goedemorgen* and *welterusten* said, and does the hour drift? | Morning greetings around 7 to 9, goodnights around 22 to 23. No drift: bedtimes are habits, not moods. |
| **Q3** Which *other* markers change over time? | Two candidates from the literature on couples' language: terms of endearment (*schat, liefje, lieverd*) become more frequent, and the share of *we*-words (*we, wij, ons, samen*) against *I*-words (*ik, mij, mijn*) rises. Both should hold for *both* authors, or it is one person's habit rather than a change in the conversation. |

What would count as an answer: a weekly rate, per 100 messages, with the ramp-up weeks
(too few messages to make a rate) removed, and the same rate split per author.

## 2. Data: what is one row, and what is missing?

One row is one WhatsApp message. Three things to know about this table before trusting a
count made from it:

1. **The clock.** The export carries local wall-clock time, and the preprocessor labels that
   as UTC. The hour in the data is therefore already the hour on the phone. Do *not*
   convert to `Europe/Amsterdam`: that would shift every evening two hours later.
2. **Merged rows.** Some exported lines contain a second timestamp in the middle, marked
   by a left-to-right character. Those are two or more messages the parser folded into one
   row. They inflate message length and hide a few timestamps. Counted below, not fixed.
3. **Media.** Photos, videos and stickers arrive as the text `image omitted`. They are
   rows that say nothing, so any rate *per message* has them in the denominator.

In [ ]:
own = load_own_chat()
own["author"] = own.author.map({name: humanize(name) for name in own.author.unique()})

print(f"{len(own):,} messages, {own.author.nunique()} authors, "
      f"{own.timestamp.min():%Y-%m-%d} to {own.timestamp.max():%Y-%m-%d}")
own[["timestamp", "author"]].head(3)

A `TransformBase` step is the unit of work here: one class, one method, one column. Two
calendar steps first, because `TimeFeatures` has no fractional hour (needed for a median
bedtime) and no datetime week start (needed to put weeks on a date axis).

In [ ]:
class FractionalHour(TransformBase):
    """Hour of day as a float, so 21:45 becomes 21.75 and a median bedtime makes sense."""

    def transform(self, data: pd.DataFrame, column: str, feature: str = "fhour") -> pd.DataFrame:
        ts = pd.to_datetime(data[column])
        data[feature] = ts.dt.hour + ts.dt.minute / 60
        return data


class WeekStart(TransformBase):
    """The Monday that starts each row's week, as a datetime, so weeks sit on a date axis."""

    def transform(self, data: pd.DataFrame, column: str, feature: str = "week") -> pd.DataFrame:
        ts = pd.to_datetime(data[column])
        if ts.dt.tz is not None:
            ts = ts.dt.tz_localize(None)
        data[feature] = ts.dt.to_period("W").dt.start_time
        return data

### The data-quality steps are pipeline steps too

`RegexFeature` in `count` mode counts embedded timestamps; in `has` mode it flags media
rows. Both are columns, so the same `Filter` syntax can drop them later if a question needs
it. The left-to-right mark (`\u200e`) WhatsApp puts before system text is the most
reliable signature of a merged row. It goes into the pattern as a Python string rather than
a regex escape, because the regex engine behind pandas' string columns does not read `\u`.

In [ ]:
LRM = "\u200e"  # the left-to-right mark, spelled out because it is invisible in a pattern

quality = (
    Pipeline()
    .add(RegexFeature, name="merged", column="message",
         pattern=LRM + r"\[\d{2}/\d{2}/\d{4}", feature="n_merged", mode="count")
    .add(RegexFeature, name="media", column="message",
         pattern=r"(?i)\b(?:image|video|audio|sticker|gif) omitted", feature="is_media", mode="has")
)
checked = quality.apply(own)

merged_rows = (checked.n_merged > 0).sum()
print(f"{merged_rows:,} rows hold {checked.n_merged.sum():,} extra messages folded into them "
      f"({merged_rows / len(checked):.1%} of rows)")
print(f"{checked.is_media.sum():,} media rows ({checked.is_media.mean():.1%})")

## 3. Features: one regex per question

Every pattern starts with `(?i)`, the inline flag for case-insensitive matching, so
*Mis je* and *mis je* are the same phrase without a lowercase step. `\b` is a word
boundary: `\bmis\b` matches *mis* and not *misschien*. `(?:...)` groups without capturing,
which matters in `extract` mode, where the *one* capturing group is what lands in the column.

| feature | pattern, in words |
|---|---|
| `miss_variant` | *ik mis je*, *mis je*, *miste je*, *mis jou*, *ik mis ...* and *je mist* (the reply), extracted so the variants can be counted. Deliberately **not** *gemist*: "die had ik gemist" is about a message, not a person. |
| `is_morning` | *goedemorgen*, *goeiemorgen*, *mogge*, or a message that opens with *morgen* |
| `is_night` | *welterusten*, *truste(n)*, *slaap lekker*, *slaap zacht*, *lekker slapen*, *slaapwel*, *goedenacht* |
| `is_endearment` | a noun of address: *schat*, *schatje*, *liefje*, *lieverd(je)*. Not *lief* or *lieve*: those are adjectives half the time. |
| `n_we`, `n_ik` | counts of *we/wij/ons/onze/samen* and *ik/mij/me/mijn* per message |

In [ ]:
MISS = r"(?i)((?:ik )?mis(?:te)? (?:je|jou)|ik mis|je mist)"
MORNING = r"(?i)\bgoe(?:de|ie)\s?morgen\b|\bmogge\b|^\s*morgen\b"
NIGHT = (r"(?i)\bwelterusten\b|\btrust?en?\b|\bslaap (?:lekker|zacht)\b"
         r"|\blekker slapen\b|\bslaapwel\b|\bgoe(?:de|ie)\s?nacht\b")
ENDEARMENT = r"(?i)\b(?:schat(?:je)?|liefje|lieverd(?:je)?)\b"
WE_WORDS = r"(?i)\b(?:we|wij|ons|onze|samen)\b"
IK_WORDS = r"(?i)\b(?:ik|mij|me|mijn)\b"

features = (
    Pipeline()
    .add(TimeFeatures, column="timestamp", features=["date", "hour"])
    .add(FractionalHour, column="timestamp")
    .add(WeekStart, column="timestamp")
    .add(RegexFeature, name="miss", column="message", pattern=MISS,
         feature="miss_variant", mode="extract")
    .add(RegexFeature, name="morning", column="message", pattern=MORNING,
         feature="is_morning", mode="has")
    .add(RegexFeature, name="night", column="message", pattern=NIGHT,
         feature="is_night", mode="has")
    .add(RegexFeature, name="endearment", column="message", pattern=ENDEARMENT,
         feature="is_endearment", mode="has")
    .add(RegexFeature, name="we", column="message", pattern=WE_WORDS,
         feature="n_we", mode="count")
    .add(RegexFeature, name="ik", column="message", pattern=IK_WORDS,
         feature="n_ik", mode="count")
)
enriched = features.apply(checked)
enriched["is_miss"] = enriched.miss_variant.notna()
print(features)

### Read the leftovers before the counts

`extract` mode logs its own coverage above: a little over one in a hundred messages. The
next check is the one everyone skips. Which variants did the pattern actually catch? A
regex that "works" can be catching the wrong thing entirely, and the only way to know is to
look at what it matched.

In [ ]:
variants = (
    Pipeline()
    .add(Filter, expr="is_miss")
    .add(CountValues, column="miss_variant")
    .apply(enriched.assign(miss_variant=enriched.miss_variant.str.lower()))
)
variants

## 4. Shape: from messages to weeks

The question is about *rates over time*, so the unit has to change from a message to a
week. One custom step does the whole regrouping: the size of each week, the mean of every
boolean (a mean of a boolean is a share, times 100 a rate per 100 messages), and the sum of
every count. A second step turns the two pronoun counts into a *we*-share.

Weeks with fewer than 100 messages are dropped: the two ramp-up weeks in late March, with
6 and 21 messages. One *mis je* in a six-message week would be a rate of 17 per 100, and
that is noise wearing the costume of a finding.

In [ ]:
class GroupProfile(TransformBase):
    """One row per group: its size, the mean of each `rates` column (x100) and the sum of each `counts` column."""

    def transform(
        self,
        data: pd.DataFrame,
        by,
        rates: list[str],
        counts: list[str] | None = None,
        per: int = 100,
    ) -> pd.DataFrame:
        grouped = data.groupby(by)
        profile = grouped.size().rename("n").to_frame()
        for column in rates:
            profile[f"{column}_rate"] = grouped[column].mean() * per
        for column in counts or []:
            profile[column] = grouped[column].sum()
        return profile.reset_index()


class ShareOf(TransformBase):
    """`feature` = a / (a + b), as a percentage: the share of `a` among the two."""

    def transform(self, data: pd.DataFrame, a: str, b: str, feature: str) -> pd.DataFrame:
        data[feature] = data[a] / (data[a] + data[b]) * 100
        return data


MIN_MESSAGES = 100

weekly_pipeline = (
    Pipeline()
    .add(GroupProfile, by="week",
         rates=["is_miss", "is_endearment", "is_morning", "is_night"],
         counts=["n_we", "n_ik"])
    .add(ShareOf, a="n_we", b="n_ik", feature="we_share")
    .add(Filter, expr=f"n >= {MIN_MESSAGES}")
)
weekly = weekly_pipeline.apply(enriched)
weekly.set_index("week").round(2)

## 5. Encoding and critique, one question at a time

### Q1: *ik mis je* per 100 messages, per week

A line, because the claim is about a trend, and the raw weekly points on it, because a
smoothed line alone hides how noisy sixteen weeks are.

In [ ]:
settings = PlotSettings(
    figsize=(11, 4),
    title="'Ik mis je' per 100 messages: below 1 in April, between 1 and 3 from May on",
    xlabel="",
    ylabel="messages with a 'mis je' variant, per 100",
    xtick_rotation=45,
)
lines = LinePlot(settings)
fig, ax = lines.plot(data=weekly, x="week", y="is_miss_rate", marker="o", color="crimson")
lines.plot_on(HorizontalLine(settings), y=weekly.is_miss_rate.mean(),
              label=f"overall {weekly.is_miss_rate.mean():.1f} per 100", linestyle=":")
_ = lines.plot_on(Annotate(settings),
              text="April: under 1 per 100",
              xy=(weekly.week.iloc[0], weekly.is_miss_rate.iloc[0]),
              xytext=(weekly.week.iloc[1], 2.8))

The first prediction holds in outline only: the rate starts below one per hundred and
settles between one and three, but it is not a climb, it is a step in late April followed
by noise. Two weeks in May and June drop back under one, and those are the two busiest
weeks in the data. That is the second prediction, so test it as a scatter rather than by
eye.

In [ ]:
scatter = PlotSettings(
    figsize=(6, 4),
    title="Busy weeks say 'mis je' less often per message",
    xlabel="messages in the week",
    ylabel="'mis je' per 100 messages",
)
points = ScatterPlot(scatter)
fig, ax = points.plot(data=weekly, x="n", y="is_miss_rate", alpha=0.8, color="crimson")
corr = weekly[["n", "is_miss_rate"]].corr().iloc[0, 1]
_ = points.plot_on(Annotate(scatter), text=f"r = {corr:.2f}",
                   xy=(weekly.n.max() * 0.8, weekly.is_miss_rate.max() * 0.9))

**Critique before claiming.** A rate per message has the week's volume in its
denominator, so a negative correlation between "rate" and "volume" is partly built in: the
same twelve *mis je* messages give a lower rate in a busier week. The honest version of
this claim needs a denominator that is not the volume, for example *mis je* per *day*, or
per day the two were not together. That last column does not exist in the export. This
stays a hypothesis, and section 7 says what data it would take.

### Q2: at what hour are *goedemorgen* and *welterusten* said?

Counts per hour of day, morning and night side by side on one scale. `FillRange` is a
one-line honesty step: an hour with zero greetings must appear as a zero bar, not vanish
from the axis.

In [ ]:
class FillRange(TransformBase):
    """Make every value in `values` present in `column`, with `fill` where the data has no row."""

    def transform(self, data: pd.DataFrame, column: str, values, fill=0) -> pd.DataFrame:
        return (
            data.set_index(column)
            .reindex(list(values), fill_value=fill)
            .rename_axis(column)
            .reset_index()
        )


def hourly(flag: str) -> pd.DataFrame:
    return (
        Pipeline()
        .add(Filter, expr=flag)
        .add(CountValues, column="hour")
        .add(FillRange, column="hour", values=range(24))
        .add(SortValues, column="hour")
        .apply(enriched)
    )


greetings = pd.concat(
    [hourly("is_morning").assign(greeting="goedemorgen"),
     hourly("is_night").assign(greeting="welterusten")]
)
print(greetings.groupby("greeting").n.sum())

hours = PlotSettings(
    figsize=(11, 3.5),
    title="Greetings sit where you would put them: mornings at 7 to 8, goodnights at 21 to 22",
    subplot_xlabels=["hour of day", "hour of day"],
    subplot_ylabels=["messages", ""],
    sharey=True,
)
fig, axes = FacetPlot(hours).plot(BarPlot(hours), data=greetings, by="greeting",
                                  order=["goedemorgen", "welterusten"],
                                  x="hour", y="n", color="steelblue")

Two things to read off. The hours are where the prediction put them. And the *counts* are
small: a few dozen morning greetings over four months of daily chatting. People do not type
*goedemorgen* every morning; they type *hoi* or start mid-sentence. So the explicit greeting
is a poor instrument for "when does the day start". A better one is already in the data: the
first and last message of each day, which exist on every day, greeting or not.

Weekly medians of both, next to the median hour of the explicit *welterusten*.

In [ ]:
daily = (
    Pipeline()
    .add(GroupAgg, by="date", column="fhour", agg="min", feature="first_message")
    .add(WeekStart, column="date")
    .apply(enriched)
)
daily["last_message"] = enriched.groupby("date").fhour.max().values

bedtime = (
    Pipeline()
    .add(GroupAgg, by="week", column="first_message", agg="median")
    .apply(daily)
    .merge(Pipeline().add(GroupAgg, by="week", column="last_message", agg="median").apply(daily))
    .merge(Pipeline()
           .add(Filter, expr="is_night")
           .add(GroupAgg, by="week", column="fhour", agg="median", feature="welterusten_hour")
           .apply(enriched), how="left")
    .merge(weekly[["week"]])
)

clock = PlotSettings(
    figsize=(11, 4),
    title="The day's edges do not move: first message ~8:30, last ~22:00, 'welterusten' at the same hour",
    xlabel="",
    ylabel="hour of day (weekly median)",
    xtick_rotation=45,
)
lines = LinePlot(clock)
fig, ax = lines.plot(data=bedtime, x="week", y="last_message", marker="o",
                     color="navy", label="last message of the day")
lines.plot_on(LinePlot(clock), data=bedtime, x="week", y="welterusten_hour", marker="s",
              color="crimson", label="'welterusten' said at")
lines.plot_on(LinePlot(clock), data=bedtime, x="week", y="first_message", marker="o",
              color="darkorange", label="first message of the day")
ax.set_ylim(0, 24)
ax.legend(loc="center right")

Prediction Q2 survives: no drift worth a sentence. The last message of the day sits
between 21:00 and 23:00 every week, and *welterusten* is said at the same hour, which is
what a goodnight is for. The morning edge wanders between 7 and 10 and is the noisier of
the two, as you would expect from a line that mixes workdays and weekends; its first point
sits at 14:00 because in that week the chat was not yet a daily habit. The red line rests
on two to thirteen greetings a week; the blue line rests on every day. When two instruments
agree, trust the one with more data.

### Q3: two candidate features, per author from the start

Endearments per 100 messages and the *we*-share, per week. The prediction said "for both
authors", so the plot is per author from the start: `FacetPlot` draws the same line once
per level of `author`, on a shared y-axis so the two panels can be compared by shape.

In [ ]:
by_author = (
    Pipeline()
    .add(GroupProfile, by=["week", "author"], rates=["is_endearment"], counts=["n_we", "n_ik"])
    .add(ShareOf, a="n_we", b="n_ik", feature="we_share")
    .add(Filter, expr="n >= 50")
    .apply(enriched)
)

endear = PlotSettings(
    figsize=(11, 3.5),
    title="Terms of endearment go from about 1 to over 10 per 100 messages, for both",
    subplot_xlabels=["", ""],
    subplot_ylabels=["endearments per 100 messages", ""],
    sharey=True,
    xtick_rotation=45,
)
fig, axes = FacetPlot(endear).plot(LinePlot(endear), data=by_author, by="author",
                                   x="week", y="is_endearment_rate", marker="o", color="crimson")

In [ ]:
we = PlotSettings(
    figsize=(11, 3.5),
    title="'We' gains on 'ik' for one author; the other was already there",
    subplot_xlabels=["", ""],
    subplot_ylabels=["we-words as % of we + ik words", ""],
    sharey=True,
    xtick_rotation=45,
)
fig, axes = FacetPlot(we).plot(LinePlot(we), data=by_author, by="author",
                               x="week", y="we_share", marker="o", color="navy")

**Critique.** The endearment panels rise together: flat near one per hundred through April
and May, then a step up in June that holds, for both. That is a change in the conversation
and not one person's tic. The *we*-share is a different picture. One author climbs from
zero to about ten percent; the other starts near ten and stays there, spike in May aside.
Half of prediction Q3 fails on the plot that was built to test it, which is what the
prediction was for. Whatever the pooled line does, "the couple talks more in *we*" is not
what happened: one person caught up with the other.

## 6. Verification: how would I know it is real?

Three checks, in increasing order of what they cost.

**Split the sample.** Done above: per author. A trend that holds in two independent halves
of the data is harder to get by chance than one that holds in the pooled series.

**Shuffle the labels.** If the week labels carried no information, how often would a
straight line through sixteen random weekly rates be as steep as the one observed? Shuffle
the rates across weeks two thousand times and count.

In [ ]:
def slope(y: np.ndarray) -> float:
    """Slope of a straight line through y, per week."""
    return float(np.polyfit(np.arange(len(y)), y, 1)[0])


def permutation_p(y: np.ndarray, n_shuffles: int = 2000, seed: int = 0) -> tuple[float, float]:
    """Observed slope and the share of shuffles at least that steep in either direction."""
    rng = np.random.default_rng(seed)
    observed = slope(y)
    null = np.array([slope(rng.permutation(y)) for _ in range(n_shuffles)])
    return observed, float((np.abs(null) >= abs(observed)).mean())


for column in ["is_endearment_rate", "we_share", "is_miss_rate"]:
    observed, p = permutation_p(weekly[column].to_numpy())
    print(f"{column:<20} slope {observed:+.3f} per week   p = {p:.3f}")

Only the endearment slope is one that shuffling almost never produces. The pooled
*we*-share slope comes up by chance about one time in ten, and the plot already said why:
one author's climb is diluted by the other's flat line. The *mis je* slope sits at the
edge, which matches its picture: a step and then noise. Two of three predicted trends do
not clear the bar, and that is the honest result, not a failure of the method.

**Try to make the finding disappear.** The most useful verification is the one that
kills a feature you liked. Here is one that looked like a finding for about an hour: the
number of *new words* entering the conversation each week that then stick (used at least
five times). It falls off a cliff after April, which reads like "the shared vocabulary
settled".

In [ ]:
class NewWordsPerWeek(TransformBase):
    """Words first seen in each week that recur at least `min_count` times in the whole chat."""

    def transform(
        self, data: pd.DataFrame, column: str, by: str, min_count: int = 5, feature: str = "new_words"
    ) -> pd.DataFrame:
        tokens = data[column].fillna("").str.lower().str.findall(r"[a-zà-ÿ]{3,}")
        first_seen: dict[str, object] = {}
        counts: dict[str, int] = {}
        for week, words in zip(data[by], tokens):
            for word in set(words):
                counts[word] = counts.get(word, 0) + 1
                first_seen.setdefault(word, week)
        first = pd.Series(first_seen)
        sticky = first[pd.Series(counts)[first.index] >= min_count]
        return sticky.value_counts().sort_index().rename(feature).rename_axis(by).reset_index()


innovation = Pipeline().add(NewWordsPerWeek, column="message", by="week")

observed = innovation.apply(enriched).merge(weekly[["week"]])
shuffled = innovation.apply(
    enriched.assign(message=enriched.message.sample(frac=1, random_state=1).to_numpy())
).merge(weekly[["week"]])

vocab = PlotSettings(
    figsize=(11, 4),
    title="New words per week: the same curve when the messages are shuffled, so it is not behaviour",
    xlabel="",
    ylabel="new words that recur 5+ times",
    xtick_rotation=45,
)
lines = LinePlot(vocab)
fig, ax = lines.plot(data=observed, x="week", y="new_words", marker="o", color="crimson",
                     label="messages in real order")
lines.plot_on(LinePlot(vocab), data=shuffled, x="week", y="new_words", marker="o",
              color="grey", linestyle="--", label="messages in random order")
ax.legend()

Same curve. Any text produces fewer new words the longer it runs: after ten thousand words
most common words have already appeared, whatever the order they were said in. The
shuffle is a *baseline model* of that, and the residual between the two lines is roughly
zero. The feature measured the size of the corpus, not the couple. Dropped.

That is the whole verification stage in one picture: before claiming a pattern is about
people, ask what the pattern would look like if the people were not there.

## 7. What to write down

1. **One row** is one message, including 391 media rows and 394 rows that hold 692
   messages folded into them; the clock is local wall-clock time even though the dtype
   says UTC.
2. **Q1** A *mis je* variant appears in about 1.5 of every 100 messages: under 1 in April,
   between 1 and 3 afterwards, with no trend that beats shuffling. The busiest weeks have
   the lowest rate, but that comparison is confounded by its own denominator. To test it
   properly I need a column the export does not have: which days the two were in the same
   place.
3. **Q2** Mornings at 7 to 8, goodnights at 21 to 22, and neither hour moves across four
   months. The explicit greeting is rare; the first and last message of the day measure the
   same thing on every day and agree with it.
4. **Q3** Terms of endearment rise from about 1 to over 10 per 100 messages in both
   authors, and survive a permutation test. The *we*-share rises in one author only, and
   the pooled trend does not survive the test. Half a prediction confirmed, half refuted.
5. **A feature that failed.** New words per week is the corpus getting longer, not the
   conversation changing. It was cheap to check and it would have been the most quotable
   line in the notebook.

Every regex above is a decision that could have gone another way: *gemist* is out, *lief*
is out, *me* is in. Change one and rerun `features.apply` to see how much the answer moves.
If it moves a lot, the finding was about the pattern, not the people.